In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

/home/adellorto/Projects/MiniGPT/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [ ]:
from tokenizer import CharachterLevelTokenizer, TiktokenTokenizer, MinbpeTokenizer

In [ ]:
# Choose dataset: "shakespeare" or "text8"
DATASET = "shakespeare"

# Max characters for MinBPE training (None = use all). Useful for large datasets like text8.
MINBPE_MAX_CHARS = None

In [ ]:
if DATASET == "shakespeare":
    with open('data/input.txt', 'r', encoding='utf-8') as f:
        text = f.read()
elif DATASET == "text8":
    from huggingface_hub import hf_hub_download
    import json
    sentences_path = hf_hub_download(
        repo_id="roshbeed/text8-dataset",
        filename="text8_sentences.json",
    )
    with open(sentences_path, 'r') as f:
        data = json.load(f)
    text = ' '.join(data['sentences'])
else:
    raise ValueError(f"Unknown dataset: {DATASET}")

print(f"Dataset: {DATASET} | Length: {len(text):,} chars")
print(text[:200])

In [ ]:
char_tokenizer = CharachterLevelTokenizer(text)
tiktoken_tokenizer = TiktokenTokenizer(text)
minbpe_tokenizer = MinbpeTokenizer(text, max_chars=MINBPE_MAX_CHARS)

print(f"CharTokenizer vocab size:       {len(char_tokenizer.vocab)}")
print(f"Tiktoken compact vocab size:    {len(tiktoken_tokenizer.vocab)}")
print(f"Tiktoken full vocab size:       {tiktoken_tokenizer.encoding.n_vocab}")
print(f"MinBPE compact vocab size:      {len(minbpe_tokenizer.vocab)}")

In [5]:
base_params = {
    'batch_size' : 32,
    'block_size' : 8,
    'max_iters' : 3000,
    'eval_interval' : 300,
    'learning_rate' : 1e-2,
    'eval_iters' : 200  
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(42)

In [6]:
def get_batch(data, params, device):
    ix = torch.randint(len(data) - params['block_size'], (params['batch_size'],))
    x = torch.stack([data[i:i+params['block_size']] for i in ix])
    y = torch.stack([data[i+1:i+params['block_size']+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [7]:
@torch.no_grad()
def estimate_loss(model, data, params, device):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(params['eval_iters'])
        for k in range(params['eval_iters']):
            X, Y = get_batch(data[split], params, device)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [8]:
# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


In [ ]:
from tqdm import tqdm

def run_experiment(tokenizer, name, text):
    # Encode full text first
    encoded = torch.tensor(tokenizer.encode(text), dtype=torch.long)

    vocab_size = len(tokenizer.vocab)
    params = {**base_params, 'vocab_size': vocab_size}

    # Train/val split
    n = int(0.9 * len(encoded))
    data = {'train': encoded[:n], 'val': encoded[n:]}

    # Model + optimizer
    torch.manual_seed(42)
    model = BigramLanguageModel(vocab_size).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=params['learning_rate'])

    print(f"\n=== {name} | vocab_size={vocab_size} | tokens={len(encoded)} ===")
    history = []
    pbar = tqdm(range(params['max_iters'] + 1), desc=name)
    for iter in pbar:
        if iter % params['eval_interval'] == 0:
            losses = estimate_loss(model, data, params, device)
            pbar.set_postfix(train=f"{losses['train']:.4f}", val=f"{losses['val']:.4f}")
            tqdm.write(f"  step {iter:4d}: train {losses['train']:.4f}  val {losses['val']:.4f}")
            history.append({'step': iter, 'train': losses['train'].item(), 'val': losses['val'].item()})
        if iter < params['max_iters']:
            xb, yb = get_batch(data['train'], params, device)
            _, loss = model(xb, yb)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

    return model, history

In [ ]:
char_model, char_history = run_experiment(char_tokenizer, "CharacterLevel", text)

In [ ]:
tiktoken_model, tiktoken_history = run_experiment(tiktoken_tokenizer, "Tiktoken (cl100k)", text)

In [ ]:
minbpe_model, minbpe_history = run_experiment(minbpe_tokenizer, "MinBPE", text)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=False)

for ax, history, name in zip(axes, [char_history, tiktoken_history, minbpe_history], ["CharacterLevel", "Tiktoken (cl100k)", "MinBPE"]):
    steps = [h['step'] for h in history]
    ax.plot(steps, [h['train'] for h in history], label='train')
    ax.plot(steps, [h['val'] for h in history], label='val', linestyle='--')
    ax.set_title(name)
    ax.set_xlabel('step')
    ax.set_ylabel('loss')
    ax.legend()

plt.suptitle('Bigram Model: Loss Comparison by Tokenizer')
plt.tight_layout()
plt.show()

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)

print("=== CharacterLevel generation ===")
tokens = char_model.generate(context, max_new_tokens=500)[0].tolist()
print(char_tokenizer.decode(tokens))

print("\n=== Tiktoken generation ===")
tokens = tiktoken_model.generate(context, max_new_tokens=500)[0].tolist()
print(tiktoken_tokenizer.decode(tokens))

print("\n=== MinBPE generation ===")
tokens = minbpe_model.generate(context, max_new_tokens=500)[0].tolist()
print(minbpe_tokenizer.decode(tokens))